# Neural Hydrology — A/B/C Run on Colab

Single-click run of the locked A/B/C protocol from `idea1.md` on Google Colab.

## Configuration

Two variables to adjust if needed (Cell 2):
- `DRIVE_CAMELS_PATH` — where your `camels_us/` folder is on Google Drive
- `MODE` — `'demo'` (1 seed × 3 conditions, ~1.5 hr) or `'full'` (5 seeds × 3 conditions, ~7-8 hr on T4)

## How to use

1. **Runtime → Change runtime type → T4 GPU** (free Colab gives this)
2. Verify `DRIVE_CAMELS_PATH` in Cell 2 matches where you put `camels_us/` on Drive
3. Pick `MODE` in Cell 2 (`'demo'` recommended for first run)
4. **Runtime → Run all**
5. When done, results are at `/content/drive/MyDrive/neural_hydrology_runs/` and lightweight summary CSVs are git-pushed back to your repo (Cell 12)

Notebook is **idempotent** — if a session disconnects, just Run All again. Already-completed seeds are skipped.

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Configuration

**Adjust `DRIVE_CAMELS_PATH`** if your folder is elsewhere on Drive. The notebook auto-detects common locations first.

In [ ]:
import os

# === USER CONFIG ===
GITHUB_URL = 'https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH = ''  # leave empty for auto-detection; or set explicitly e.g.
                          #   '/content/drive/MyDrive/datasets/camels_us'
                          #   '/content/drive/MyDrive/neural_hydrology/datasets/camels_us'
MODE = 'demo'  # 'demo' (1 seed, ~1.5 hr) or 'full' (5 seeds, ~7-8 hr on T4)
# ====================

# Auto-detect camels_us if not explicitly set
AUTO_DETECT_CANDIDATES = [
    '/content/drive/MyDrive/datasets/camels_us',
    '/content/drive/MyDrive/neural_hydro/datasets/camels_us',
    '/content/drive/MyDrive/neural_hydrology/datasets/camels_us',
    '/content/drive/MyDrive/camels_us',
    '/content/drive/MyDrive/data/camels_us',
]
if not DRIVE_CAMELS_PATH:
    for cand in AUTO_DETECT_CANDIDATES:
        if os.path.isdir(cand):
            DRIVE_CAMELS_PATH = cand
            print(f'Auto-detected camels_us at: {cand}')
            break
    else:
        raise RuntimeError(
            'Could not auto-detect camels_us folder on Drive. Tried:\n  ' +
            '\n  '.join(AUTO_DETECT_CANDIDATES) +
            '\n\nSet DRIVE_CAMELS_PATH explicitly above to the folder containing camels_us.')
else:
    assert os.path.isdir(DRIVE_CAMELS_PATH), f'{DRIVE_CAMELS_PATH} not found'
    print(f'Using camels_us at: {DRIVE_CAMELS_PATH}')

# Verify a known sub-path inside camels_us
topo_file = os.path.join(DRIVE_CAMELS_PATH, 'camels_attributes_v2.0', 'camels_topo.txt')
assert os.path.isfile(topo_file), f'Expected {topo_file} — does the folder have CAMELS contents?'
print(f'Verified camels_topo.txt present.')

# Persistent runs dir on Drive
DRIVE_RUNS = '/content/drive/MyDrive/neural_hydrology_runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
print(f'Runs dir: {DRIVE_RUNS}')

# Seed selection
SEEDS = [42] if MODE == 'demo' else [11, 13, 17, 19, 23]
print(f'\nMODE = {MODE}; SEEDS = {SEEDS}')
print(f'Estimated wall-clock: ~{1.5 if MODE == "demo" else 7.5} hr on T4')

## Cell 3 — Clone the repo from GitHub

In [ ]:
REPO_DIR = '/content/nh'

import shutil
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
print()
!git log --oneline -n 3

## Cell 4 — Install dependencies

Pinning `numpy<2` because torch 2.2 is incompatible with numpy 2.x.

In [ ]:
%cd {REPO_DIR}
!pip install -q -e . "numpy<2" pynhd networkx 2>&1 | tail -3

# Verify
import numpy as np, torch
x = torch.from_numpy(np.array([1.0]))
print(f'numpy {np.__version__}  torch {torch.__version__}  CUDA: {torch.cuda.is_available()}')

## Cell 5 — Symlink data and runs from Drive

In [ ]:
%cd {REPO_DIR}

# datasets/camels_us -> Drive folder
REPO_DATA = os.path.join(REPO_DIR, 'datasets', 'camels_us')
os.makedirs(os.path.dirname(REPO_DATA), exist_ok=True)
if os.path.islink(REPO_DATA) or os.path.isdir(REPO_DATA):
    !rm -rf {REPO_DATA}
os.symlink(DRIVE_CAMELS_PATH, REPO_DATA)
print(f'datasets/camels_us -> {DRIVE_CAMELS_PATH}')

# runs/ -> Drive (so checkpoints survive session end)
REPO_RUNS = os.path.join(REPO_DIR, 'runs')
if os.path.islink(REPO_RUNS) or os.path.isdir(REPO_RUNS):
    !rm -rf {REPO_RUNS}
os.symlink(DRIVE_RUNS, REPO_RUNS)
print(f'runs/ -> {DRIVE_RUNS}')

# Quick verify
!ls datasets/camels_us | head -5

## Cell 6 — GPU check

In [ ]:
!nvidia-smi -L
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> select T4 GPU.')
print(f'GPU: {torch.cuda.get_device_name(0)}, '
       f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 7 — Generate per-seed YAML configs for Condition A

In [ ]:
%cd {REPO_DIR}
CONFIG_DIR = os.path.join(REPO_DIR, 'experiments', 'configs', '_seed_configs')
os.makedirs(CONFIG_DIR, exist_ok=True)

BASE_CONFIG = open('experiments/configs/lstm_component0_baseline.yaml').read()
for seed in SEEDS:
    cfg = BASE_CONFIG.replace('experiment_name: lstm_component0_baseline',
                                f'experiment_name: A_baseline_seed{seed}')
    cfg = cfg.replace('device: cpu', 'device: cuda:0')
    if 'seed:' not in cfg:
        cfg += f'\nseed: {seed}\n'
    out = os.path.join(CONFIG_DIR, f'A_seed{seed}.yaml')
    with open(out, 'w') as f:
        f.write(cfg)
print(f'Wrote {len(SEEDS)} seed configs')

## Cell 8 — Smoke test (1 epoch on the 23-basin pilot to catch setup errors)

~3 min on T4. If this fails, the rest will fail — fix here first.

In [ ]:
%cd {REPO_DIR}
# Train a 30-ep run-05-equivalent so train_graph_component0.py has a baseline-run to point at
# Skip if already exists
import glob
if not glob.glob(f'{REPO_DIR}/runs/05_lstm_23basin_strong_baseline'):
    print('Training 23-basin baseline as a one-time prerequisite for B/C cfg + scaler...')
    !python neuralhydrology/nh_run.py train --config-file experiments/configs/lstm_study_network_strong.yaml 2>&1 | tail -3
    # Rename to numbered
    pilot = sorted(glob.glob(f'{REPO_DIR}/runs/lstm_study_network_strong_*'))[-1]
    !mv {pilot} {REPO_DIR}/runs/05_lstm_23basin_strong_baseline
else:
    print('23-basin pilot baseline already exists.')

# Quick smoke test of graph-LSTM training
!python experiments/training/train_graph_component0.py \
    --variant warm \
    --seed 42 \
    --smoke-test \
    --no-warm-start \
    --basin-file experiments/basin_lists/study_network_basins.txt \
    --edge-file topology_analysis/phase1_network_discovery/outputs/study_network_edges.csv \
    --baseline-run runs/05_lstm_23basin_strong_baseline 2>&1 | tail -5

## Cell 9 — Condition A (NH cudalstm baseline, no graph) × seeds

In [ ]:
%cd {REPO_DIR}
import time, glob
for seed in SEEDS:
    if glob.glob(f'{REPO_DIR}/runs/A_baseline_seed{seed}_*/model_epoch030.pt'):
        print(f'[skip] A seed={seed} already complete')
        continue
    cfg = f'{CONFIG_DIR}/A_seed{seed}.yaml'
    print(f'\n=== Condition A — seed={seed} ===')
    t0 = time.time()
    !python neuralhydrology/nh_run.py train --config-file {cfg} 2>&1 | tail -3
    print(f'    {(time.time() - t0)/60:.1f} min')

## Cell 10 — Condition B (graph-LSTM with topology features, no message passing)

In [ ]:
%cd {REPO_DIR}
BASELINE_FOR_BC = sorted(glob.glob(f'{REPO_DIR}/runs/A_baseline_seed*/'))[0]
print(f'Using {BASELINE_FOR_BC} for B/C cfg + scaler')

for seed in SEEDS:
    if glob.glob(f'{REPO_DIR}/runs/graph_c0_topology_features_seed{seed}_*/test_metrics.csv'):
        print(f'[skip] B seed={seed} already complete')
        continue
    print(f'\n=== Condition B — seed={seed} ===')
    t0 = time.time()
    !python experiments/training/train_graph_component0.py \
        --variant topology_features \
        --seed {seed} --no-warm-start --epochs 30 \
        --baseline-run {BASELINE_FOR_BC} 2>&1 | tail -3
    print(f'    {(time.time() - t0)/60:.1f} min')

## Cell 11 — Condition C (full graph-LSTM with edges + message passing)

In [ ]:
%cd {REPO_DIR}
for seed in SEEDS:
    if glob.glob(f'{REPO_DIR}/runs/graph_c0_warm_seed{seed}_*/test_metrics.csv'):
        print(f'[skip] C seed={seed} already complete')
        continue
    print(f'\n=== Condition C — seed={seed} ===')
    t0 = time.time()
    !python experiments/training/train_graph_component0.py \
        --variant warm \
        --seed {seed} --no-warm-start --epochs 30 \
        --baseline-run {BASELINE_FOR_BC} 2>&1 | tail -3
    print(f'    {(time.time() - t0)/60:.1f} min')

## Cell 12 — Aggregate results + push back to GitHub

Writes `summary.json` and `per_basin_per_seed.csv` to `experiments/analysis_outputs/abc_publication/`, then attempts a git commit + push so you can `git pull` locally and ask CRS to interpret.

In [ ]:
%cd {REPO_DIR}
import json, re
import pandas as pd
import numpy as np
from pathlib import Path

def load_metrics(pattern):
    out = {}
    for p in sorted(glob.glob(pattern)):
        m = re.search(r'seed(\d+)', p)
        if not m:
            continue
        df = pd.read_csv(p, dtype={'basin': str})
        out[int(m.group(1))] = df.set_index('basin')['NSE'].to_dict()
    return out

results = {
    'A_baseline': load_metrics(f'{REPO_DIR}/runs/A_baseline_seed*/test/model_epoch030/test_metrics.csv'),
    'B_topology_features': load_metrics(f'{REPO_DIR}/runs/graph_c0_topology_features_seed*/test_metrics.csv'),
    'C_graph_messages': load_metrics(f'{REPO_DIR}/runs/graph_c0_warm_seed*/test_metrics.csv'),
}

summary = {}
for label, r in results.items():
    if not r:
        continue
    medians = sorted(np.median(list(d.values())) for d in r.values())
    summary[label] = {
        'n_seeds': len(medians),
        'seeds': sorted(r.keys()),
        'median_NSE_per_seed': medians,
        'cross_seed_median': float(np.median(medians)),
        'cross_seed_std': float(np.std(medians)) if len(medians) > 1 else 0.0,
    }
print(json.dumps(summary, indent=2))

OUT_DIR = Path(REPO_DIR) / 'experiments' / 'analysis_outputs' / 'abc_publication'
OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
rows = [{'condition': label, 'seed': s, 'basin': b, 'NSE': v}
        for label, r in results.items() for s, d in r.items() for b, v in d.items()]
if rows:
    pd.DataFrame(rows).to_csv(OUT_DIR / 'per_basin_per_seed.csv', index=False)

if {'A_baseline', 'C_graph_messages'}.issubset(summary):
    a = summary['A_baseline']['cross_seed_median']
    c = summary['C_graph_messages']['cross_seed_median']
    print(f'\n*** C - A median NSE delta = {c - a:+.3f}')
    if 'B_topology_features' in summary:
        b = summary['B_topology_features']['cross_seed_median']
        print(f'    B - A = {b - a:+.3f}    C - B = {c - b:+.3f}')

# Push results back to GitHub
print('\n=== Pushing results to GitHub ===')
!git config user.email "colab@example.com"
!git config user.name "colab"
!git checkout -b colab-results-$(date +%Y%m%d-%H%M%S) 2>&1 | tail -1
!git add experiments/analysis_outputs/abc_publication/ 2>&1
!git commit -m "A/B/C results from Colab ({MODE} mode, seeds {SEEDS})" 2>&1 | tail -2
print('\nIf the next push fails with auth, you need to use a GitHub PAT.')
print('Run this once locally to get a token: gh auth token')
print('Then in Colab: !git remote set-url origin https://<token>@github.com/Op-2005/neural_hydro.git')
!git push -u origin HEAD 2>&1 | tail -5

## Done

If push succeeded: locally do `git pull` (or `git fetch && git checkout colab-results-<timestamp>`), then in chat say *`crs interpret abc results`*.

If push failed (no auth): the `summary.json` and `per_basin_per_seed.csv` are still on Drive at `/content/drive/MyDrive/neural_hydrology_runs/` — copy them to your local repo manually.